# DeepSeek-OCR on Google Colab

1. Select **Runtime → Change runtime type → GPU**.
2. Run the cells from top to bottom through the smoke test. The first run downloads the model weights (about 6.7 GB).
3. The notebook clones and checks [this public repository](https://github.com/ubaid-148/deeksheekocr), then runs OCR on a simple test image. The last cell lets you upload your own image.

Inference follows the [upstream vLLM DeepSeek-OCR recipe](https://docs.vllm.ai/projects/recipes/en/latest/DeepSeek/DeepSeek-OCR.html). Colab GPU availability varies; if no GPU is assigned, reconnect with a GPU runtime.

In [ ]:
import shutil
import subprocess

if not shutil.which('nvidia-smi'):
    raise RuntimeError('Select a GPU runtime: Runtime → Change runtime type → GPU')
subprocess.run(['nvidia-smi'], check=True)

## Clone and check the public repository

The repository has Python scripts rather than a compiled build target. This cell checks every Python source file without creating build files.

In [ ]:
import ast
from pathlib import Path

REPO_URL = 'https://github.com/ubaid-148/deeksheekocr.git'
REPO_DIR = Path('/content/deeksheekocr')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

sources = sorted(REPO_DIR.rglob('*.py'))
if not sources:
    raise RuntimeError('No Python files found in the public repository')
for source in sources:
    ast.parse(source.read_text(encoding='utf-8'), filename=str(source))
print(f'Checked {len(sources)} Python files in {REPO_DIR}')
subprocess.run(['git', '-C', str(REPO_DIR), 'status', '--short', '--branch'], check=True)

## Install the GPU runtime

vLLM supplies the model runtime and compatible PyTorch packages. This notebook uses vLLM's CUDA backend selection, so the pinned `requirements.txt` for the repository's older Transformers example is not installed here. On the first install, this cell restarts the Python session to clear previously loaded packages such as Pillow. After it reconnects, run the notebook again from the top.

In [ ]:
import importlib.metadata
import sys
from IPython import get_ipython

try:
    vllm_version = importlib.metadata.version('vllm')
    installed_now = False
except importlib.metadata.PackageNotFoundError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'uv'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--system', '-U', 'vllm', '--torch-backend=auto'], check=True)
    vllm_version = importlib.metadata.version('vllm')
    installed_now = True

pillow_check = subprocess.run(
    [sys.executable, '-c', 'from PIL import Image, ImageText'],
    capture_output=True, text=True,
)
if pillow_check.returncode != 0:
    error_lines = pillow_check.stderr.strip().splitlines()
    print('Repairing Pillow:', error_lines[-1] if error_lines else 'import failed')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall', 'pillow'], check=True)
    installed_now = True
cached_pil_typing = sys.modules.get('PIL._typing')
if cached_pil_typing is not None and not hasattr(cached_pil_typing, '_Ink'):
    print('An older Pillow module is still loaded in this session')
    installed_now = True
print('vLLM:', vllm_version)
if installed_now:
    print('Dependencies changed. Restarting the Python session; rerun the notebook from the top after reconnecting.')
    get_ipython().kernel.do_shutdown(restart=True)
else:
    print('Dependencies ready')

## Load DeepSeek-OCR

T4 GPUs need float16. A100 and newer GPUs use bfloat16. The GPU probe runs in a child process so the notebook kernel has not initialized CUDA before vLLM starts its worker.

In [ ]:
import os

probe = subprocess.check_output([
    sys.executable, '-c',
    'import torch; assert torch.cuda.is_available(); print(*torch.cuda.get_device_capability(0), sep=".")'
], text=True).strip()
gpu_major, gpu_minor = map(int, probe.splitlines()[-1].split('.'))
if (gpu_major, gpu_minor) < (7, 5):
    raise RuntimeError('vLLM needs an NVIDIA GPU with compute capability 7.5 or newer')
model_dtype = 'float16' if gpu_major < 8 else 'bfloat16'
print('GPU compute capability:', f'{gpu_major}.{gpu_minor}', 'model dtype:', model_dtype)

from vllm import LLM, SamplingParams
from vllm.model_executor.models.deepseek_ocr import NGramPerReqLogitsProcessor

llm = LLM(
    model='deepseek-ai/DeepSeek-OCR',
    dtype=model_dtype,
    max_model_len=4096,
    max_num_seqs=1,
    gpu_memory_utilization=0.85,
    enforce_eager=True,
    enable_prefix_caching=False,
    mm_processor_cache_gb=0,
    logits_processors=[NGramPerReqLogitsProcessor],
)
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=1024,
    extra_args={
        'ngram_size': 30,
        'window_size': 90,
        'whitelist_token_ids': {128821, 128822},
    },
    skip_special_tokens=False,
)
PROMPT = '<image>\n<|grounding|>Convert the document to markdown.'
print('Model ready')

## Smoke test

In [ ]:
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display

sample = Image.new('RGB', (1000, 300), 'white')
draw = ImageDraw.Draw(sample)
font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 44)
draw.text((40, 30), 'INVOICE', font=font, fill='black')
draw.text((40, 115), 'Item: Notebook', font=font, fill='black')
draw.text((40, 200), 'Total: $42.50', font=font, fill='black')
display(sample)
result = llm.generate(
    [{'prompt': PROMPT, 'multi_modal_data': {'image': sample}}],
    sampling_params,
)[0].outputs[0].text
if not result.strip():
    raise RuntimeError('OCR returned empty text')
print('\nOCR result:\n', result)
Path('/content/deepseek_ocr_smoke.md').write_text(result, encoding='utf-8')

## Run OCR on your own image (optional)

Run this cell separately after the smoke test. Upload a PNG, JPG, or JPEG file. The Markdown result is saved and downloaded.

In [ ]:
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one PNG, JPG, or JPEG image')
image_name = next(iter(uploaded))
if Path(image_name).suffix.lower() not in {'.png', '.jpg', '.jpeg'}:
    raise ValueError('Only PNG, JPG, and JPEG are supported in this cell')
with Image.open(image_name) as opened:
    user_image = opened.convert('RGB')
output = llm.generate(
    [{'prompt': PROMPT, 'multi_modal_data': {'image': user_image}}],
    sampling_params,
)[0].outputs[0].text
output_file = Path('/content') / f'{Path(image_name).stem}_ocr.md'
output_file.write_text(output, encoding='utf-8')
print(output)
files.download(str(output_file))